In [ ]:
import requests
from sqlitesearch import TextSearchIndex

# 1. Descargar los datos crudos desde la API del profesor
print("Descargando datos del curso...")
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course["path"]}'
    course_response = requests.get(course_url)
    course_data = course_response.json()
    documents.extend(course_data)

# Filtramos solo los del curso de LLMs para nuestro agente
docs_llm = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']
print(f"Total de documentos de LLM Zoomcamp: {len(docs_llm)}")

# 2. Guardar en SQLite (Base de datos persistente)
print("Creando base de datos persistente (faq.db)...")
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db' # Magia de la persistencia
)

# Insertamos los documentos en la base de datos
for doc in docs_llm:
    index.add(doc)

# Cerramos la conexión para asegurar que se guarden los datos en el disco
index.close()
print("¡Ingesta completada! Los datos están a salvo en 'faq.db'.")

Descargando datos del curso...
Total de documentos de LLM Zoomcamp: 79
Creando base de datos persistente (faq.db)...
¡Ingesta completada! Los datos están a salvo en 'faq.db'.


In [ ]:
from sqlitesearch import TextSearchIndex
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# 1. Nos conectamos a nuestra base de datos persistente La que creaMOS en el paso 2)
index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

# 2. Definimos nuestra herramienta con Buenas Prácticas (Type Hints y Docstrings)
# NO escribimos ningún esquema JSON gigante aquí
def search(query: str) -> dict:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

# 3. Inicializamos ToyAIKit y dejamos que él auto-genere el esquema JSON
agent_tools = Tools()
agent_tools.add_tool(search)

# print("Herramientas auto-generadas:", agent_tools.get_tools())

# 4. El Prompt del Desarrollador (Las instrucciones del Agente)
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
"""

# 5. Configuramos la Interfaz Visual y el Bucle Agéntico (Runner)
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-4o-mini') 
)

print("\n¡Agente 2026 Listo! Escribe tu pregunta abajo. Escribe 'stop' para salir.")

# 6. Arrancamos el chat interactivo
runner.run()

In [2]:
from gitsource import GithubRepositoryDataReader

# 1. Leemos los archivos del repositorio usando el commit exacto que pide la tarea
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",           # Commit fijado para que todos tengan los mismos datos
    allowed_extensions={"md"},     # Solo archivos Markdown
    filename_filter=lambda path: "/lessons/" in path, # Solo archivos dentro de carpetas /lessons/
)

files = reader.read()

# 2. Parseamos los archivos a una lista de diccionarios
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

# 3. Respondemos la Q1
print(f"Respuesta Q1 - Número total de lecciones: {len(documents)}")

Respuesta Q1 - Número total de lecciones: 72


In [3]:
from minsearch import Index

# 1. Configuramos el índice según las instrucciones de la tarea
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

# 2. Indexamos los 72 documentos de la Q1
print("Indexando documentos...")
index.fit(documents)

# 3. Realizamos la búsqueda
query = "How does the agentic loop keep calling the model until it stops?"
print(f"Buscando: '{query}'\n")

search_results = index.search(
    query=query,
    num_results=5
)

# 4. Imprimimos el nombre de archivo del primer resultado (nuestra respuesta a la Q2)
primer_resultado = search_results[0]['filename']
print(f"Respuesta Q2 - El filename del primer resultado es:")
print(f"--> {primer_resultado}")

Indexando documentos...
Buscando: 'How does the agentic loop keep calling the model until it stops?'

Respuesta Q2 - El filename del primer resultado es:
--> 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
import urllib.request
from openai import OpenAI

# 1. Importamos la clase original
from rag_helper import RAGBase

# 2. Creamos una NUEVA clase que hereda de RAGBase
class HomeworkRAG(RAGBase):
    
    # Sobreescribimos la búsqueda para quitar el filtro 'course'
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results
        )
    
    # Sobreescribimos cómo se arma el contexto
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append("")
        return "\n".join(lines).strip()
    
    # Sobreescribimos el llamado al LLM
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response.output_text, response.usage.input_tokens
        
    # Sobreescribimos la función principal rag
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        
        answer, input_tokens = self.llm(prompt)
        return answer, input_tokens

# 3. Inicializamos nuestro asistente
openai_client = OpenAI()

assistant = HomeworkRAG(
    index=index, 
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

# 4. Hacemos la consulta de la Q3
query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = assistant.rag(query)

print(f"Respuesta Q3 - Tokens de entrada (Input Tokens): {input_tokens}")

Respuesta Q3 - Tokens de entrada (Input Tokens): 7126


In [6]:
from gitsource import chunk_documents

# 1. Aplicamos la fragmentación (chunking) a nuestros 72 documentos originales
# size=2000: Cada pedazo tendrá hasta 2000 caracteres
# step=1000: La ventana avanza de 1000 en 1000 (hay 1000 caracteres de superposición)
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Imprimimos el total de chunks generados (Nuestra respuesta a la Q4)
print(f"Respuesta Q4 - Número de chunks generados: {len(chunks)}")

Respuesta Q4 - Número de chunks generados: 295


In [7]:
# 1. Creamos un NUEVO índice, pero esta vez indexamos los CHUNKS
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
print("Indexando los 295 chunks...")
index_chunks.fit(chunks)

# 2. Inicializamos el asistente RAG apuntando al NUEVO índice
assistant_chunked = HomeworkRAG(
    index=index_chunks, # Usamos el índice de los pedacitos
    llm_client=openai_client,
    model='gpt-5.4-mini'
)

# 3. Hacemos la misma pregunta
query = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens_chunked = assistant_chunked.rag(query)

print(f"\nRespuesta Q5:")
print(f"Tokens SIN chunking (Q3): {input_tokens}")
print(f"Tokens CON chunking (Q5): {input_tokens_chunked}")

Indexando los 295 chunks...

Respuesta Q5:
Tokens SIN chunking (Q3): 7126
Tokens CON chunking (Q5): 2309


In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# 1. Creamos la herramienta usando el índice de pedacitos (index_chunks)
def search(query: str) -> dict:
    """
    Search the course lessons database for entries matching the given query.
    """
    # Usamos index_chunks (de la Q5) en lugar de index (de la Q2)
    return index_chunks.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

# 2. Instrucciones exactas de la tarea (obligan a buscar múltiples veces)
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

# 3. Configuramos la UI y el Runner
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

# 4. Hacemos la pregunta de la tarea
query = "How does the agentic loop work, and how is it different from plain RAG?"
print(f"Preguntando: '{query}'\n")

# Ejecutamos un solo turno del bucle
result = runner.loop(prompt=query, callback=callback)

# 5. Contamos automáticamente de forma segura
conteo = 0
for msg in result.all_messages:
    if hasattr(msg, 'type') and msg.type == 'function_call':
        conteo += 1
    elif isinstance(msg, dict) and msg.get('type') == 'function_call':
        conteo += 1

print(f"\nRespuesta Q6 - El agente llamó a la herramienta 'search': {conteo} veces.")

Preguntando: 'How does the agentic loop work, and how is it different from plain RAG?'

-> Response received


-> Response received



Respuesta Q6 - El agente llamó a la herramienta 'search': 4 veces.
